# Florian Ewing
# 2.2.2025

This project applies machine learning to predict whether a chemical compound extends organism lifespan. Using curated biological and chemical data, we train a binary classifier to distinguish lifespan-extending compounds from non-extending ones.

- **Task**: Binary classification (lifespan extension: Yes / No)
- **Primary Model**: Random Forest Classifier
- **Domain**: Computational biology / drug discovery / aging research

---

## Objectives

- Build a supervised machine learning model to predict lifespan-extending compounds.
- Engineer biologically meaningful and chemical features.
- Evaluate model performance using standard classification metrics.

---

## Dataset

### Primary Dataset: DrugAge

- **Source**: DrugAge Database  
  https://genomics.senescence.info/drugs/
- **Description**:  
  A curated dataset of chemical compounds annotated with their effects on organism lifespan.
- **Labels**:
  - `1`: Lifespan increased
  - `0`: Lifespan not increased

The dataset is used to extract:
- Compound identifiers
- Lifespan outcome labels

In [6]:
from pathlib import Path
print("CWD:", Path.cwd())

CWD: c:\Users\flori\Documents\GitHub\Ora-Biomedical-Student-Machine-Learning-Project\notebooks


# Accessing the Data
# Exploratory Data Analysis

In [7]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DGIDB_PATH = PROJECT_ROOT / "data" / "DGIdb"

categories_df = pd.read_csv(DGIDB_PATH / "categories.tsv", sep="\t")
drugs_df = pd.read_csv(DGIDB_PATH / "drugs.tsv", sep="\t")
genes_df = pd.read_csv(DGIDB_PATH / "genes.tsv", sep="\t")
interactions_df = pd.read_csv(DGIDB_PATH / "interactions.tsv", sep="\t")

print("Loaded:")
print("Categories:", categories_df.shape)
print("Drugs:", drugs_df.shape)
print("Genes:", genes_df.shape)
print("Interactions:", interactions_df.shape)

Loaded:
Categories: (32795, 4)
Drugs: (81572, 9)
Genes: (80234, 6)
Interactions: (98239, 13)


In [8]:
categories_df.head()

,name,name-2,source_db_name,source_db_version
0,PXR,NUCLEAR HORMONE RECEPTOR,BaderLab,Feb-14
1,LXRA,NUCLEAR HORMONE RECEPTOR,BaderLab,Feb-14
2,SHP,NUCLEAR HORMONE RECEPTOR,BaderLab,Feb-14
3,DAX,NUCLEAR HORMONE RECEPTOR,BaderLab,Feb-14
4,REV-ERBA,NUCLEAR HORMONE RECEPTOR,BaderLab,Feb-14


In [9]:
drugs_df.head()

,drug_claim_name,nomenclature,concept_id,drug_name,approved,immunotherapy,anti_neoplastic,source_db_name,source_db_version
0,BRAF(V600E) Kinase Inhibitor RO5212054,Primary Drug Name,ncit:C92591,BRAF(V600E) KINASE INHIBITOR RO5212054,False,False,True,NCIt,24.02d
1,clonidine hydrochloride,Primary Drug Name,rxcui:142432,CLONIDINE HYDROCHLORIDE,True,False,False,RxNorm,20240401
2,clonidine,Primary Drug Name,rxcui:142432,CLONIDINE HYDROCHLORIDE,True,False,False,RxNorm,20240401
3,Clonidine,Primary Drug Name,rxcui:142432,CLONIDINE HYDROCHLORIDE,True,False,False,NCIt,24.02d
4,Clonidine Hydrochloride,Primary Drug Name,rxcui:142432,CLONIDINE HYDROCHLORIDE,True,False,False,NCIt,24.02d


In [10]:
genes_df.head()

,gene_claim_name,nomenclature,concept_id,gene_name,source_db_name,source_db_version
0,NGFIBA,NCBI Gene Name,NaN,NaN,BaderLab,Feb-14
1,NGFIBB,NCBI Gene Name,NaN,NaN,BaderLab,Feb-14
2,DAX,NCBI Gene Name,NaN,NaN,BaderLab,Feb-14
3,REV-ERBA,NCBI Gene Name,NaN,NaN,BaderLab,Feb-14
4,COUP2,NCBI Gene Name,NaN,NaN,BaderLab,Feb-14


In [11]:
interactions_df.head()

,gene_claim_name,gene_concept_id,gene_name,interaction_source_db_name,interaction_source_db_version,interaction_type,interaction_score,drug_claim_name,drug_concept_id,drug_name,approved,immunotherapy,anti_neoplastic
0,CYP2D6,hgnc:2625,CYP2D6,DTC,9/2/20,NaN,0.017709,RACLOPRIDE,ncit:C152139,RACLOPRIDE,False,False,False
1,PPARG,hgnc:9236,PPARG,DTC,9/2/20,NaN,0.840123,KALOPANAX-SAPONIN F,chembl:CHEMBL1833984,CHEMBL:CHEMBL1833984,False,False,False
2,ATAD5,hgnc:25752,ATAD5,DTC,9/2/20,NaN,0.177992,RO-5-3335,chembl:CHEMBL91609,CHEMBL:CHEMBL91609,False,False,False
3,RGS4,hgnc:10000,RGS4,DTC,9/2/20,NaN,0.034319,"3,4-DICHLOROISOCOUMARIN",drugbank:DB04459,"3,4-DICHLOROISOCOUMARIN",False,False,False
4,MAPK1,hgnc:6871,MAPK1,DTC,9/2/20,NaN,0.050007,WITHAFERIN A,iuphar.ligand:13097,WITHAFERIN A,False,False,False


In [12]:
DRUGAGE_PATH = PROJECT_ROOT / "data" / "DrugAgeDataset" / "drugage.csv"
drugage_df = pd.read_csv(DRUGAGE_PATH)
print("DrugAge:", drugage_df.shape)
drugage_df.head()

DrugAge: (3423, 15)


,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id
0,Ethanol,Drosophila mojavensis,A300,4%,NaN,NaN,81.29,NaN,NaN,NaN,Male,NaN,NaN,No,13369
1,Ethanol,Drosophila mojavensis,A300,2%,NaN,NaN,50.66,NaN,NaN,NaN,Male,NaN,NaN,No,13369
2,Ethanol,Drosophila mojavensis,A300,4%,NaN,NaN,71.03,NaN,NaN,NaN,Female,NaN,NaN,No,13369
3,Ethanol,Drosophila mojavensis,A300,2%,NaN,NaN,123.26,NaN,NaN,NaN,Female,NaN,NaN,No,13369
4,Ethanol,Drosophila mojavensis,A512d,4%,NaN,NaN,20.88,NaN,NaN,NaN,Male,NaN,NaN,No,13369


In [13]:
lifespan_cols = [
    "avg_lifespan_change_percent",
    "avg_lifespan_significance",
    "max_lifespan_change_percent",
    "max_lifespan_significance"
]

drugage_df[lifespan_cols].isna().sum()


avg_lifespan_change_percent      51
avg_lifespan_significance       211
max_lifespan_change_percent    2417
max_lifespan_significance      3151
dtype: int64

In [14]:
def show_data_by_species(species_name: str):
    # Standardize species names by stripping whitespace
    drugage_df['species'] = drugage_df['species'].str.strip()
    species_name = species_name.strip()
    
    # All columns, lifespan/age columns clustered in the middle
    all_cols = [
        "compound_name",
        "species",
        "strain",
        "dosage",
        "age_at_initiation",
        "treatment_duration",
        "avg_lifespan_change_percent",
        "avg_lifespan_significance",
        "max_lifespan_change_percent",
        "max_lifespan_significance",
        "gender",
        "weight_change_percent",
        "weight_change_significance",
        "ITP",
        "pubmed_id"
    ]
    
    if species_name not in drugage_df['species'].unique():
        raise ValueError("Species not found in dataset. Try again.")
    
    species_df = drugage_df[drugage_df['species'] == species_name]
    return species_df[all_cols]

In [15]:
show_data_by_species("Mus musculus")

,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id
140,Butylated hydroxytoluene,Mus musculus,BaLB/c,7500 ppm food,2.6 months,until death,25.0,S,0.0,NS,Female,NaN,NaN,No,448040
141,Butylated hydroxytoluene,Mus musculus,BaLB/c,7500 ppm food,2.6 months,until death,30.0,S,0.0,NS,Male,NaN,NaN,No,448040
142,Butylated hydroxytoluene,Mus musculus,BaLB/c,7500 ppm food,2 months,until death,14.0,S,0.0,NS,Female,NaN,NaN,No,448040
143,Butylated hydroxytoluene,Mus musculus,BaLB/c,7500 ppm food,2 months,until death,22.0,S,0.0,NS,Male,NaN,NaN,No,448040
144,Butylated hydroxytoluene,Mus musculus,BaLB/c,7500 ppm food,2 months,2.6 months,8.0,S,0.0,NS,Female,NaN,NaN,No,448040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3418,Canagliflozin,Mus musculus,UM-HET3,180 ppm food,16 months,until death,14.0,S,4.0,S,Male,-9.0,S,Yes,38753230
3419,Alpha-ketoglutarate,Mus musculus,UM-HET3,20000 ppm food,18 months,until death,1.0,NS,-5.0,NS,Female,8.0,NS,Yes,38753230
3420,Alpha-ketoglutarate,Mus musculus,UM-HET3,20000 ppm food,18 months,until death,3.0,NS,-3.0,NS,Male,0.0,NS,Yes,38753230
3421,X203 (anti-IL-11),Mus musculus,C57BL/6 J,40 ppm bodyweight monthly intraperitoneal inje...,17.2 months,until death,25.0,S,NaN,NaN,Female,NaN,NaN,No,39020175


# Feature Engineering
### 1. Biological Features (Initial Focus)

- **Gene Ontology (GO) terms** for proteins targeted by each compound
- **Source**: Drug–Gene Interaction Database (DGIdb)  
  https://dgidb.org/downloads
- **Approach**:
  - Map compounds to target genes/proteins
  - Encode associated GO terms as features (e.g., binary or frequency-based vectors)

The `get_compound_to_gene_mapping` function generates a clean mapping between chemical compounds and their target genes using DGIdb data. It merges the `interactions` table with the `drugs` and `genes` tables to standardize compound and gene names, selects only the relevant columns, removes any missing values, and drops duplicates. The output is a DataFrame with two columns — `compound_name` and `target_gene` — which can be directly used for downstream biological feature engineering, such as linking compounds to Gene Ontology terms.


In [16]:
def get_compound_to_gene_mapping(drugs_df, genes_df, interactions_df):
    """
    Returns a DataFrame mapping compounds to target genes.
    
    Args:
        drugs_df (DataFrame): DGIdb drugs.tsv
        genes_df (DataFrame): DGIdb genes.tsv
        interactions_df (DataFrame): DGIdb interactions.tsv
    
    Returns:
        DataFrame: columns ['compound_name', 'target_gene']
    """
    # Merge interactions with drugs to get canonical compound names
    compound_gene_df = interactions_df.merge(
        drugs_df[['drug_name']], left_on='drug_name', right_on='drug_name', how='left'
    )
    
    # Merge with genes to get canonical gene names
    compound_gene_df = compound_gene_df.merge(
        genes_df[['gene_name']], left_on='gene_name', right_on='gene_name', how='left'
    )
    
    # Select relevant columns and drop rows with missing data
    compound_gene_df = compound_gene_df[['drug_name', 'gene_name']].dropna()
    
    # Rename for clarity
    compound_gene_df.rename(columns={'drug_name': 'compound_name', 'gene_name': 'target_gene'}, inplace=True)
    
    # Remove duplicates (optional)
    compound_gene_df = compound_gene_df.drop_duplicates()
    
    return compound_gene_df

In [ ]:
drugs_sample = drugs_df.head(20)
genes_sample = genes_df.head(20)
interactions_sample = interactions_df.head(20)

# Test the mapping function
compound_gene_mapping_test = get_compound_to_gene_mapping(drugs_sample, genes_sample, interactions_sample)

print("Compound → Gene mapping shape (test):", compound_gene_mapping_test.shape)
print(compound_gene_mapping_test)

Compound → Gene mapping shape (test): (20, 2)
                   compound_name target_gene
0                     RACLOPRIDE      CYP2D6
1           CHEMBL:CHEMBL1833984       PPARG
2             CHEMBL:CHEMBL91609       ATAD5
3        3,4-DICHLOROISOCOUMARIN        RGS4
4                   WITHAFERIN A       MAPK1
5                 ANGIOTENSIN II       AGTR1
6                      NERATINIB         KDR
7                       CEP-9722       PARP1
8                      ETORPHINE       NUDT2
9   COMPOUND 8E [PMID: 24432909]        DTNB
10                      GDC-0339         F10
11                     RYANODINE        RYR3
12                      IMATINIB         KIT
13                CLODRONIC ACID        CCL4
14                      PLATINUM        CSF2
15     OXYTETRACYCLINE ANHYDROUS       BAZ2B
16                  BENZENETHIOL         JUN
17           CHEMBL:CHEMBL590168       EHMT2
18                     CISPLATIN     CSNK2A3
19                    ARBUTAMINE       ADRB1


In [ ]:
#PLACEHOLDER: gene_go_df https://geneontology.org/

The `build_go_feature_matrix` function transforms compound–gene relationships into a machine-learning-ready binary matrix based on Gene Ontology (GO) terms. It merges a compound-to-gene mapping with gene-to-GO annotations, then constructs a crosstab where rows represent compounds, columns represent GO terms, and entries are 1 if the compound targets a gene associated with that GO term (0 otherwise). The resulting matrix encodes the biological processes, molecular functions, or cellular components influenced by each compound, providing features suitable for predictive modeling.

In [ ]:
def build_go_feature_matrix(compound_gene_df, gene_go_df):
    """
    Builds a binary feature matrix of compounds vs GO terms.

    Args:
        compound_gene_df (DataFrame): ['compound_name', 'target_gene']
        gene_go_df (DataFrame): ['target_gene', 'go_term']

    Returns:
        DataFrame: rows=compounds, columns=GO terms, values=0/1
    """
    # Merge compound-gene mapping with gene → GO terms
    compound_go_df = compound_gene_df.merge(
        gene_go_df, left_on='target_gene', right_on='target_gene', how='left'
    )
    
    # Create binary matrix
    binary_matrix = pd.crosstab(
        compound_go_df['compound_name'], 
        compound_go_df['go_term']
    )
    
    # Ensure all values are binary (0/1)
    binary_matrix[binary_matrix > 0] = 1
    
    return binary_matrix


In [19]:
# Take small samples for testing
compound_gene_sample = compound_gene_mapping_test.head(10)  # from previous test
gene_go_sample = gene_go_df.head(20)  # your gene → GO mapping

# Test the GO feature matrix function
go_feature_matrix_test = build_go_feature_matrix(compound_gene_sample, gene_go_sample)

print("GO feature matrix shape (test):", go_feature_matrix_test.shape)
print(go_feature_matrix_test)


NameError: name 'gene_go_df' is not defined